In this exercise, we will predict the number of applications received
using the other variables in the College data set.

College 資料集（ISLR 內建）：
Apps：申請人數（預測的目標 Y）</br>
Accept</br>
Enroll</br>
Top10perc</br>
Top25perc</br>
F.Undergrad</br>
Room.Board</br>
Books</br>
Personal</br>
PhD</br>
Terminal</br>
S.F.Ratio</br>
perc.alumni</br>
Expend</br>
Grad.Rate</br>
Private</br>

In [11]:
from ISLP import load_data
import numpy as np
import pandas as pd

college = load_data('College')


# 將 Private (Yes/No) 轉為 0/1
college = pd.get_dummies(college, drop_first=True)
college

,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate,Private_Yes
0,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60,1
1,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56,1
2,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54,1
3,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59,1
4,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
772,2197,1515,543,4,26,3089,2029,6797,3900,500,1200,60,60,21.0,14,4469,40,0
773,1959,1805,695,24,47,2849,1107,11520,4960,600,1250,73,75,13.3,31,9189,83,1
774,2097,1915,695,34,61,2793,166,6900,4200,617,781,67,75,14.4,20,8323,49,1
775,10705,2453,1317,95,99,5217,83,19840,6510,630,2115,96,96,5.8,49,40386,99,1


(a) Split the data set into a training set and a test set.

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split


X = college.drop(columns=["Apps"])
y = college["Apps"]

# 70% training, 30% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)


(b) Fit a linear model using least squares on the training set, and
report the test error obtained.

In [13]:
	
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

lm = LinearRegression()
lm.fit(X_train, y_train)

y_pred = lm.predict(X_test)
test_mse = mean_squared_error(y_test, y_pred)

print("Test MSE:", test_mse)


Test MSE: 1659682.1719134296


(c) Fit a ridge regression model on the training set, with λ chosen
by cross-validation. Report the test error obtained.

In [ ]:
from sklearn.linear_model import RidgeCV #自動用交叉驗證選 λ 的 Ridge
import numpy as np

lambdas = np.logspace(-3, 3, 50) #0.001 - 1000 , logspace(-3, 3, 50) = 在 log10 空間平均取 50 點。
ridge = RidgeCV(alphas=lambdas, scoring='neg_mean_squared_error', cv=10) #告訴模型，你要試這 50 個 λ。 建立一個 RidgeCV 模型：
# alphas=lambdas：告訴模型，你要試這 50 個 λ。
# scoring='neg_mean_squared_error'：用 MSE 當評估指標（因為 sklearn 定義越大越好，所以用負的 MSE）。
# cv=10：做 10-fold cross-validation。
ridge.fit(X_train, y_train)

y_pred = ridge.predict(X_test)
test_mse_ridge = mean_squared_error(y_test, y_pred)

print("Ridge Test MSE:", test_mse_ridge)
print("Best Lambda:", ridge.alpha_)


Ridge Test MSE: 1652355.4706191502
Best Lambda: 3.5564803062231287


(d) Fit a lasso model on the training set, with λ chosen by cross-
validation. Report the test error obtained, along with the num-
ber of non-zero coefficient estimates.

In [ ]:
# (d) LASSO + Cross-Validation，外加「非零係數個數」
from sklearn.linear_model import LassoCV

lasso = LassoCV(cv=10, random_state=0)
lasso.fit(X_train, y_train)

y_pred = lasso.predict(X_test)
test_mse_lasso = mean_squared_error(y_test, y_pred)

print("LASSO Test MSE:", test_mse_lasso)
print("Best Lambda:", lasso.alpha_)

nonzero = sum(lasso.coef_ != 0)
print("Number of non-zero coefficients:", nonzero)


(e) Fit a PCR model on the training set, with M chosen by cross-
validation. Report the test error obtained, along with the value
of M selected by cross-validation.

In [15]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
import numpy as np

M_values = range(1, X_train.shape[1] + 1)
kf = KFold(n_splits=10, shuffle=True, random_state=0)

cv_errors = []

for M in M_values:
    mse_list = []
    for train_idx, val_idx in kf.split(X_train):
        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        pca = PCA(n_components=M)
        X_tr_pca = pca.fit_transform(X_tr)
        X_val_pca = pca.transform(X_val)

        lin = LinearRegression()
        lin.fit(X_tr_pca, y_tr)

        pred = lin.predict(X_val_pca)
        mse_list.append(mean_squared_error(y_val, pred))
    cv_errors.append(np.mean(mse_list))

best_M = M_values[np.argmin(cv_errors)]

print("Best M selected by CV:", best_M)

pca = PCA(n_components=best_M)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

lin = LinearRegression()
lin.fit(X_train_pca, y_train)

y_pred = lin.predict(X_test_pca)
test_mse_pcr = mean_squared_error(y_test, y_pred)

print("PCR Test MSE:", test_mse_pcr)


Best M selected by CV: 17
PCR Test MSE: 1659682.171913419
